# Endividamento Municipal e Dependência do FPM - Análise com Pandas

IFTO – Campus Palmas

CST em Sistemas para Internet

**Disciplina:** Fundamentos de Programação Científica (2026/2)

**Aluno:** Marcos Ribeiro Guida

**Etapa 2 do projeto final:** filtros, cruzamentos, agrupamentos e indicadores

---

### Tema e problema
O **FPM** (Fundo de Participação dos Municípios) é a principal receita da maioria das prefeituras. A LRF e a Resolução nº 40/2001 do Senado limitam a **Dívida Consolidada Líquida (DCL)** a **120% da Receita Corrente Líquida (RCL)**, com alerta a partir de **108%**.

**Hipótese:** municípios mais dependentes do FPM têm menor endividamento (DCL/RCL), pois têm menos capacidade própria de pagamento e menos acesso a crédito.

### Origem dos dados

| Base | Fonte | Chave | Informações principais |
|---|---|---|---|
| **A – FPM** | Tesouro Transparente (CSV) | código SIAFI | FPM recebido por mês, 2015–2025 |
| **B – RGF, Anexo 2** | API do SICONFI (JSON → CSV) | código IBGE | Dívida Consolidada, DCL, RCL e % DCL/RCL em 31/12/2025 |
| **C – DCA, Anexo I-C** | API do SICONFI (JSON → CSV) | código IBGE | Receita total, impostos, FPM e operações de crédito de 2025 |

### Análises desta etapa
1. **Filtros:** Região Norte, municípios de alta dependência e municípios pequenos endividados.
2. **Cruzamentos:** RGF × DCA (código IBGE) e SICONFI × FPM (de-para SIAFI × IBGE).
3. **Agrupamentos:** por região, por quintil de dependência, por região × porte e por ano.
4. **Indicadores:** dependência, situação da dívida, acesso a crédito e correlação dependência × dívida.
5. **Desempenho:** filtrar antes ou depois do `merge`.
6. **Relatório consolidado** exportado para Excel e CSV.

## 1. Configuração

In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

# No Colab, os CSVs ficam em Meu Drive/Projeto_Endividamento/; fora dele, na pasta atual.
PASTA = '/content/drive/MyDrive/Projeto_Endividamento/'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    PASTA = '..'

SAIDA = Path(PASTA) / 'resultados'
SAIDA.mkdir(exist_ok=True)

def caminho(arquivo):
    # Procura o arquivo na pasta do projeto, inclusive em subpastas.
    for achado in Path(PASTA).rglob(arquivo):
        return achado
    raise FileNotFoundError(f'{arquivo} não encontrado em {PASTA}')

fmt = lambda n: f'{n:,.0f}'.replace(',', '.')            # 12345 -> '12.345'
pct = lambda parte, total: f'{parte / total * 100:.1f}%'.replace('.', ',')

REGIAO = {**dict.fromkeys(['AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'], 'Norte'),
          **dict.fromkeys(['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'], 'Nordeste'),
          **dict.fromkeys(['DF', 'GO', 'MT', 'MS'], 'Centro-Oeste'),
          **dict.fromkeys(['ES', 'MG', 'RJ', 'SP'], 'Sudeste'),
          **dict.fromkeys(['PR', 'RS', 'SC'], 'Sul')}
REGIOES = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
LIMITE_ALERTA, LIMITE_LEGAL = 108, 120     # % DCL/RCL
LIMIAR_DEP = 0.40                          # "alta dependência": FPM ≥ 40% da receita

Mounted at /content/drive


## 2. Importação e preparação dos dados

| Base | Decisão na leitura | Preparação |
|---|---|---|
| A – FPM | `sep=';'`, `encoding='latin1'`, `COD_MUN` como texto (mantém zeros à esquerda) | Valores de texto → número; formato largo → longo, com coluna `data` |
| B – RGF | CSV gerado do JSON da API (tipos já corretos) | Filtros da etapa 1; `pivot` para uma linha por município |
| C – DCA | Idem | Filtros da etapa 1; `pivot`; cálculo da dependência do FPM |

**Ajustes em relação à etapa 1:** voltam a coluna `instituicao` (nome do município, necessário para ligar ao FPM) e a conta *Deduções* (para validar a DCL).

### 2.1 Leitura e verificação

In [3]:
fpm_original = pd.read_csv(caminho('fpm-por-municipio.csv'), sep=';', encoding='latin1', dtype={'COD_MUN': str})
rgf_original = pd.read_csv(caminho('siconfi_rgf_anexo02_municipios_2025.csv'), sep=';')
dca_original = pd.read_csv(caminho('siconfi_dca_anexoIC_municipios_2025.csv'), sep=';')

# Confere com os totais registrados na etapa 1
verificacao = pd.DataFrame({
    'linhas':          [len(fpm_original), len(rgf_original), len(dca_original)],
    'linhas esperadas': [66_848, 387_985, 712_990],
    'municípios':      [fpm_original['COD_MUN'].nunique(), rgf_original['cod_ibge'].nunique(),
                        dca_original['cod_ibge'].nunique()],
}, index=['A – FPM', 'B – RGF', 'C – DCA'])
assert (verificacao['linhas'] == verificacao['linhas esperadas']).all(), 'Importação incompleta'
verificacao

,linhas,linhas esperadas,municípios
A – FPM,66848,66848,5570
B – RGF,387985,387985,5527
C – DCA,712990,712990,5506


In [4]:
# Tipos das colunas que serão usadas
pd.concat({'A – FPM': fpm_original[['COD_MUN', 'Mês', '2025']].dtypes,
           'B – RGF': rgf_original[['cod_ibge', 'populacao', 'cod_conta', 'valor']].dtypes,
           'C – DCA': dca_original[['cod_ibge', 'cod_conta', 'valor']].dtypes}).to_frame('tipo').T

A – FPM                   B – RGF                               C – DCA                   
     COD_MUN      Mês    2025 cod_ibge populacao cod_conta    valor cod_ibge cod_conta    valor
tipo  object  float64  object    int64     int64    object  float64    int64    object  float64

**Tipos a corrigir:**
- No FPM, os valores (`2025` etc.) estão como texto no padrão brasileiro (`" 50.880,73 "`) e `Mês` está como `float64`. Também não há coluna de data.
- No RGF e na DCA os tipos estão corretos, mas os dados estão em formato longo (uma linha por conta), o que impede dividir uma conta por outra.

### 2.2 Base A – FPM: série mensal com data

In [5]:
ANOS = [str(a) for a in range(2015, 2026)]

def texto_para_numero(serie):
    # ' 50.880,73 ' -> 50880.73 ; '-' (sem repasse) -> NaN
    return pd.to_numeric(serie.str.strip().str.replace('.', '', regex=False)
                              .str.replace(',', '.', regex=False), errors='coerce')

fpm = fpm_original[['COD_MUN', 'Município', 'UF', 'Mês'] + ANOS].copy()
fpm[ANOS] = fpm[ANOS].apply(texto_para_numero)
fpm = fpm[fpm['COD_MUN'].notna() & (fpm['UF'] != 'DF')]               # filtros A1 e A2 da etapa 1
fpm = fpm[fpm[ANOS].notna().all(axis=1).groupby(fpm['COD_MUN']).transform('all')]   # A3

fpm_longo = (fpm.melt(id_vars=['COD_MUN', 'Município', 'UF', 'Mês'], var_name='ano', value_name='fpm')
                .rename(columns={'COD_MUN': 'cod_siafi', 'Município': 'municipio', 'UF': 'uf', 'Mês': 'mes'})
                .astype({'ano': int, 'mes': int}))
fpm_longo['data'] = pd.to_datetime(fpm_longo[['ano', 'mes']].rename(columns={'ano': 'year', 'mes': 'month'})
                                   .assign(day=1))
fpm_longo['regiao'] = pd.Categorical(fpm_longo['uf'].map(REGIAO), categories=REGIOES)

assert not fpm_longo.duplicated(['cod_siafi', 'data']).any() and fpm_longo['fpm'].notna().all()
print(f"{fmt(fpm_longo['cod_siafi'].nunique())} municípios × 132 meses = {fmt(len(fpm_longo))} linhas")
print(fpm_longo.dtypes.to_dict())

5.568 municípios × 132 meses = 734.976 linhas
{'cod_siafi': dtype('O'), 'municipio': dtype('O'), 'uf': dtype('O'), 'mes': dtype('int64'), 'ano': dtype('int64'), 'fpm': dtype('float64'), 'data': dtype('<M8[ns]'), 'regiao': CategoricalDtype(categories=['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul'], ordered=False, categories_dtype=object)}


### 2.3 Base B – RGF: dívida por município
- O `pivot()` gera erro se houver duas linhas para o mesmo município e conta, então ele também verifica duplicidade.
- A API omite as linhas sem valor. Por isso, antes de tratar a Dívida Consolidada ausente como **zero**, o código confere se a identidade **DCL = DC − Deduções** fecha com essa suposição.

In [6]:
FECHAMENTO = ['Até o 3º Quadrimestre', 'Até o 2º Semestre']            # posição em 31/12/2025
CONTAS_RGF = {'DividaConsolidada': 'dc', 'DeducoesDaDividaConsolidada': 'deducoes',
              'DividaConsolidadaLiquida': 'dcl',
              'ReceitaCorrenteLiquidaAjustadaParaCalculoDosLimitesDeEndividamento': 'rcl',
              'PercentualDaDCLSobreARCL': 'pct_dcl_rcl'}

rgf = rgf_original[rgf_original['coluna'].isin(FECHAMENTO) & rgf_original['cod_conta'].isin(CONTAS_RGF)]
divida = (rgf[['cod_ibge', 'instituicao', 'uf', 'populacao', 'periodicidade']].drop_duplicates('cod_ibge')
             .merge(rgf.pivot(index='cod_ibge', columns='cod_conta', values='valor').rename(columns=CONTAS_RGF),
                    on='cod_ibge'))

# Validações antes de preencher ausentes
identidade = (divida['dc'].fillna(0) - divida['deducoes'].fillna(0) - divida['dcl']).abs() < 1
recalculo = (divida['dcl'] / divida['rcl'] * 100 - divida['pct_dcl_rcl']).abs().max()
print(f"Ausentes: {divida[list(CONTAS_RGF.values())].isna().sum().to_dict()}")
print(f"DCL = DC − Deduções (ausente = 0): {pct(identidade.sum(), divida['dcl'].notna().sum())} dos municípios")
print(f"% DCL/RCL recalculado × informado: diferença máxima de {recalculo:.3f} p.p.")

divida[['dc', 'deducoes']] = divida[['dc', 'deducoes']].fillna(0)
divida['municipio'] = divida['instituicao'].str.replace(r'^Prefeitura Municipal d[eoa]s? | - [A-Z]{2}$', '', regex=True)
divida['regiao'] = pd.Categorical(divida['uf'].map(REGIAO), categories=REGIOES)
divida['porte'] = pd.cut(divida['populacao'], [0, 10_000, 20_000, 50_000, 100_000, 500_000, np.inf],
                         labels=['até 10 mil', '10–20 mil', '20–50 mil', '50–100 mil', '100–500 mil', '> 500 mil'])
divida['dc_rcl'] = divida['dc'] / divida['rcl'] * 100                  # dívida bruta / RCL (%)
divida = divida.drop(columns='instituicao')
divida.head(3)

Ausentes: {'dc': 450, 'deducoes': 35, 'dcl': 17, 'rcl': 0, 'pct_dcl_rcl': 17}
DCL = DC − Deduções (ausente = 0): 100,0% dos municípios
% DCL/RCL recalculado × informado: diferença máxima de 0.005 p.p.


,cod_ibge,uf,populacao,periodicidade,deducoes,dc,dcl,pct_dcl_rcl,rcl,municipio,regiao,porte,dc_rcl
0,3503406,SP,8079,Q,"-876,021.85","3,112,052.36","3,988,074.21",6.89,"57,901,532.48",Arealva,Sudeste,até 10 mil,5.37
1,3503505,SP,3616,S,"739,054.12","8,893,708.68","8,154,654.56",19.14,"42,608,516.38",Areias,Sudeste,até 10 mil,20.87
2,3503604,SP,9805,Q,"7,671,839.35","5,227,601.83","-2,444,237.52",-4.25,"57,539,402.99",Areiópolis,Sudeste,até 10 mil,9.09


- A identidade fecha em 100% dos municípios, então `dc` e `deducoes` ausentes viram 0. Os 17 municípios sem `dcl` continuam `NaN`.
- O percentual informado é igual ao recalculado, o que mostra que o dado é coerente.
- **Colunas criadas:**
  - `regiao`;
  - `porte` (faixas de população);
  - `dc_rcl` (dívida **bruta**/RCL), que complementa a DCL. A DCL fica negativa quando o caixa supera a dívida.

### 2.4 Base C – DCA: receitas e dependência do FPM

In [7]:
CONTAS_DCA = {'TotalReceitas': 'receita_total', 'RO1.1.0.0.00.0.0': 'impostos',
              'RO1.7.1.1.51.0.0': 'fpm_dca', 'RO2.1.0.0.00.0.0': 'op_credito'}

dca = dca_original[(dca_original['coluna'] == 'Receitas Brutas Realizadas') & dca_original['cod_conta'].isin(CONTAS_DCA)]
receitas = dca.pivot(index='cod_ibge', columns='cod_conta', values='valor').rename(columns=CONTAS_DCA).reset_index()
receitas.columns.name = None
print(f"Ausentes: {receitas.isna().sum()[lambda s: s > 0].to_dict()}")

receitas['op_credito'] = receitas['op_credito'].fillna(0)               # sem a linha = não contratou crédito
receitas['dep_fpm'] = receitas['fpm_dca'] / receitas['receita_total']   # fração da receita vinda do FPM
receitas.describe().T[['count', 'mean', '50%', 'min', 'max']]

Ausentes: {'impostos': 3, 'fpm_dca': 6, 'op_credito': 4397}


,count,mean,50%,min,max
cod_ibge,"5,506.00","3,259,653.04","3,147,055.50","1,100,015.00","5,222,302.00"
impostos,"5,503.00","61,697,469.14","5,651,090.51","76,110.40","68,282,433,803.83"
fpm_dca,"5,500.00","43,231,558.11","26,973,873.45","557,218.60","2,080,652,643.95"
op_credito,"5,506.00","3,011,761.75",0.00,0.00,"2,835,694,236.28"
receita_total,"5,506.00","284,030,610.09","93,543,094.25",132.01,"127,308,730,789.66"
dep_fpm,"5,500.00",0.30,0.29,0.00,0.69


- Só cerca de 1/5 dos municípios tem a linha de operações de crédito: a DCA só lista contas com valor, então ausente = **0**.
- O FPM ausente (6 casos) fica `NaN`, porque todo município recebe FPM e, nesses casos, o dado está incompleto.
- **`dep_fpm`** é a variável central da hipótese: a parcela da receita total que vem do FPM (mediana de 29%).

## 3. Seleção e filtragem dos registros

### Filtro 1 – Por categoria (`isin`): municípios da Região Norte
**Pergunta:** qual é a situação da dívida na região do Tocantins, onde o projeto é desenvolvido?

In [8]:
UFS_NORTE = [uf for uf, r in REGIAO.items() if r == 'Norte']
norte = divida.loc[divida['uf'].isin(UFS_NORTE), ['municipio', 'uf', 'populacao', 'dcl', 'pct_dcl_rcl']]

print(f'Registros: {fmt(len(norte))} de {fmt(len(divida))} municípios ({pct(len(norte), len(divida))})')
print('Por UF:', norte['uf'].value_counts().to_dict())
norte.sort_values('pct_dcl_rcl', ascending=False).head()

Registros: 433 de 5.525 municípios (7,8%)
Por UF: {'TO': 136, 'PA': 133, 'AM': 61, 'RO': 52, 'AC': 22, 'RR': 15, 'AP': 14}


,municipio,uf,populacao,dcl,pct_dcl_rcl
4358,Ipixuna do Pará,PA,29116,"633,513,776.30",244.21
4512,Itaguatins,TO,5187,"50,886,774.81",138.61
4426,Tailândia,PA,84206,"240,006,432.25",127.68
4448,Oiapoque,AP,27778,"34,172,108.43",127.37
5069,Rorainópolis,RR,29932,"167,288,682.35",107.01


**Importância:** o recorte regional permite comparar a situação local com a nacional e será um filtro do sistema final. Dos 433 municípios do Norte, 136 são do Tocantins, e **Itaguatins/TO (138,6%) passa do limite legal**.

### Filtro 2 – Por intervalo numérico (`between`): municípios de alta dependência
**Pergunta:** quais municípios recebem **40% ou mais** da receita total pelo FPM?

In [9]:
alta_dep = receitas.loc[receitas['dep_fpm'].between(LIMIAR_DEP, 1.0)]
com_fpm = receitas['dep_fpm'].notna().sum()

print(f'Registros: {fmt(len(alta_dep))} de {fmt(com_fpm)} municípios com FPM informado ({pct(len(alta_dep), com_fpm)})')
print(f"Impostos próprios / receita: {pct(alta_dep['impostos'].sum(), alta_dep['receita_total'].sum())} no grupo "
      f"× {pct(receitas['impostos'].sum(), receitas['receita_total'].sum())} no Brasil")
alta_dep.nlargest(5, 'dep_fpm')

Registros: 1.074 de 5.500 municípios com FPM informado (19,5%)
Impostos próprios / receita: 4,3% no grupo × 21,7% no Brasil


,cod_ibge,impostos,fpm_dca,op_credito,receita_total,dep_fpm
1093,2403905,"271,914.59","20,230,405.11",0.00,"29,338,823.19",0.69
2236,3103108,"184,639.13","20,611,734.60",0.00,"30,016,805.19",0.69
1113,2405900,"327,808.26","20,230,070.38",0.00,"30,154,071.87",0.67
1101,2404804,"1,200,388.22","20,230,405.11",0.00,"30,806,233.28",0.66
3424,3518008,"1,242,839.99","22,902,866.78",0.00,"35,078,740.41",0.65


**Importância:** este é o grupo "dependente do FPM" da hipótese. Os impostos próprios pesam **5 vezes menos** na receita desse grupo do que na média do país. Se a hipótese estiver certa, ele deve ter a menor dívida (testado nas seções 5 e 6).

### Filtro 3 – Múltiplas condições (`query` e `&`): municípios pequenos e endividados
**Pergunta:** quais municípios com **menos de 20 mil habitantes**, que são os mais dependentes do FPM, têm **DCL acima de 50% da RCL**?

In [10]:
pequenos_endividados = divida.query('populacao < 20000 and pct_dcl_rcl > 50')
assert pequenos_endividados.equals(divida.loc[(divida['populacao'] < 20000) & (divida['pct_dcl_rcl'] > 50)])

pequenos = (divida['populacao'] < 20000).sum()
print(f'Registros: {fmt(len(pequenos_endividados))} de {fmt(pequenos)} municípios pequenos ({pct(len(pequenos_endividados), pequenos)})')
print('Por região:', pequenos_endividados['regiao'].value_counts().to_dict())
pequenos_endividados.nlargest(5, 'pct_dcl_rcl')[['municipio', 'uf', 'populacao', 'pct_dcl_rcl']]

Registros: 167 de 3.823 municípios pequenos (4,4%)
Por região: {'Nordeste': 150, 'Sudeste': 10, 'Norte': 5, 'Centro-Oeste': 1, 'Sul': 1}


,municipio,uf,populacao,pct_dcl_rcl
3488,Caatiba,BA,6915,495.98
3620,Itapebi,BA,9779,247.07
2784,Carmópolis,SE,13904,231.71
3547,Dário Meira,BA,10025,190.75
2837,São Domingos,SE,10332,164.95


**Importância:** são as **exceções à hipótese**, e **150 dos 167 estão no Nordeste**. Isso indica que a região pesa tanto quanto a dependência. Valores acima de 200% sugerem erro de preenchimento no SICONFI. No sistema, este filtro vira uma **lista de alerta**.

## 4. Cruzamento entre os dados

| # | DataFrames | Chave | Junção |
|---|---|---|---|
| 4.1 | `divida` × `receitas` | `cod_ibge` | `outer` (diagnóstico) e `inner` (base) |
| 4.2 | `base` × municípios do FPM | nome normalizado + `uf` | `left` |
| 4.3 | `base` × FPM anual | `cod_siafi` | `left` |

Em todos os `merge`, `validate='one_to_one'` gera erro se a chave se repetir, o que protege contra duplicações.

### 4.1 Dívida (RGF) × receitas (DCA)

In [11]:
diagnostico = divida.merge(receitas, on='cod_ibge', how='outer', validate='one_to_one', indicator=True)
print(diagnostico['_merge'].value_counts().to_dict())

base = divida.merge(receitas, on='cod_ibge', how='inner', validate='one_to_one')
print(f"Base integrada: {fmt(len(base))} municípios | duplicados: {base['cod_ibge'].duplicated().sum()}")

{'both': 5491, 'left_only': 34, 'right_only': 15}
Base integrada: 5.491 municípios | duplicados: 0


O `inner` mantém os **5.491** municípios que entregaram os dois demonstrativos. Os 34 que só entregaram o RGF e os 15 que só entregaram a DCA entram nas limitações (seção 9).

### 4.2 De-para SIAFI × IBGE
O FPM usa o código SIAFI, e o SICONFI usa o IBGE. A ligação é feita pelo **nome sem acentos + UF**. Os 22 municípios com nome diferente nos dois cadastros (ex.: *Paraty* × *Parati*) foram corrigidos num dicionário.

In [12]:
def normalizar(nomes):
    # "Olho-d'Água" -> "OLHO D AGUA"
    return (nomes.str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii')
                 .str.upper().str.replace(r"[-']", ' ', regex=True).str.split().str.join(' '))

CORRECOES = {  # (UF, nome IBGE): nome SIAFI
    ('SP', 'Embu das Artes'): 'Embu', ('SP', 'Mogi Mirim'): 'Moji Mirim', ('RJ', 'Paraty'): 'Parati',
    ('RJ', 'Trajano de Moraes'): 'Trajano de Morais', ('MG', 'Brazópolis'): 'Brasópolis',
    ('MG', 'São Tomé das Letras'): 'São Thomé das Letras', ('CE', 'Itapajé'): 'Itapagé',
    ('PB', 'São Domingos'): 'São Domingos de Pombal', ('PB', 'São Vicente do Seridó'): 'Seridó',
    ('PE', 'Belém do São Francisco'): 'Belém de São Francisco', ('PE', 'Iguaracy'): 'Iguaraci',
    ('PE', 'Lagoa de Itaenga'): 'Lagoa do Itaenga', ('RN', 'Campo Grande'): 'Augusto Severo',
    ('RN', 'Boa Saúde'): 'Januário Cicco', ('RN', 'Serra Caiada'): 'Presidente Juscelino',
    ('MT', 'Poxoréu'): 'Poxoréo', ('PA', 'Eldorado do Carajás'): 'Eldorado dos Carajás',
    ('PA', 'Santa Izabel do Pará'): 'Santa Isabel do Pará', ('TO', 'Couto Magalhães'): 'Couto de Magalhães',
    ('TO', 'Tabocão'): 'Fortaleza do Tabocão', ('TO', 'São Valério'): 'São Valério da Natividade',
    ('RS', "Sant'Ana do Livramento"): 'Santana do Livramento',
}

siafi = fpm_longo[['cod_siafi', 'municipio', 'uf']].drop_duplicates('cod_siafi')
siafi['chave'] = normalizar(siafi['municipio'])
base['chave'] = normalizar(pd.Series([CORRECOES.get((uf, m), m) for uf, m in zip(base['uf'], base['municipio'])]))

base = base.merge(siafi[['cod_siafi', 'chave', 'uf']], on=['chave', 'uf'], how='left',
                  validate='one_to_one', indicator=True)
sem_par = base.loc[base['_merge'] == 'left_only', 'municipio'].tolist()
print(f"Ligados: {fmt((base['_merge'] == 'both').sum())} | sem par: {sem_par}")

# Único sem par esperado: Boa Esperança do Norte/MT, instalado em 2025 (sem série de FPM).
assert sem_par == ['Boa Esperança do Norte']
base = base[base['_merge'] == 'both'].drop(columns=['chave', '_merge']).reset_index(drop=True)

Ligados: 5.490 | sem par: ['Boa Esperança do Norte']


### 4.3 Base × série anual do FPM

In [13]:
fpm_anual = fpm_longo.pivot_table(index='cod_siafi', columns='ano', values='fpm', aggfunc='sum').add_prefix('fpm_')
antes = len(base)
base = base.merge(fpm_anual.reset_index(), on='cod_siafi', how='left', validate='one_to_one')
assert len(base) == antes and base['cod_ibge'].is_unique

# Conferência entre fontes: FPM 2025 do Tesouro (base A) × FPM 2025 declarado na DCA (base C)
razao = base['fpm_2025'] / base['fpm_dca']
print(f"Base final: {fmt(len(base))} municípios × {base.shape[1]} colunas | duplicados: 0")
print(f"FPM Tesouro ÷ FPM DCA: mediana {razao.median():.3f} (90% entre {razao.quantile(.05):.3f} e {razao.quantile(.95):.3f})")

base['quintil_dep'] = pd.qcut(base['dep_fpm'], 5, labels=['Q1 (menor)', 'Q2', 'Q3', 'Q4', 'Q5 (maior)'])

Base final: 5.490 municípios × 30 colunas | duplicados: 0
FPM Tesouro ÷ FPM DCA: mediana 0.822 (90% entre 0.821 e 0.823)


- Nenhum `merge` duplicou registros: a base final tem **uma linha por município**.
- A razão Tesouro/DCA é **0,82 em praticamente todos os municípios**, o que confirma que o de-para ligou os municípios certos. A diferença é o desconto do FUNDEB: o Tesouro informa o FPM líquido (metadados da base A) e a DCA informa o bruto. Por isso a dependência usa o FPM da DCA, e a série do Tesouro é usada para a evolução no tempo.
- **Contribuição:** só o cruzamento permite relacionar dependência (DCA), dívida (RGF) e trajetória do FPM (Tesouro) de um mesmo município.
- `quintil_dep` divide os municípios em 5 grupos de mesmo tamanho, do menos (Q1) ao mais dependente (Q5).

## 5. Agrupamento e agregação

A **mediana** é a medida principal, porque a DCL/RCL tem valores extremos (até 1.333%) que distorcem a média.

### 5.1 Por categoria: região

In [14]:
por_regiao = (base.groupby('regiao', observed=True)
                  .agg(municipios=('cod_ibge', 'count'),
                       dep_fpm_mediana=('dep_fpm', 'median'),
                       dcl_rcl_mediana=('pct_dcl_rcl', 'median'),
                       dc_rcl_mediana=('dc_rcl', 'median'))
                  .assign(dep_fpm_mediana=lambda d: d['dep_fpm_mediana'] * 100)
                  .sort_values('dep_fpm_mediana', ascending=False))
por_regiao

,municipios,dep_fpm_mediana,dcl_rcl_mediana,dc_rcl_mediana
regiao,,,,
Sudeste,1660,31.48,-11.34,5.12
Nordeste,1771,31.01,12.50,16.78
Sul,1175,28.95,-14.16,8.13
Centro-Oeste,463,23.24,-8.10,6.99
Norte,421,23.07,-1.33,7.99


Sudeste e Nordeste têm a **mesma dependência** (cerca de 31%), mas **dívidas opostas**: a DCL/RCL mediana é −11% no Sudeste (caixa maior que a dívida) e **+12,5% no Nordeste**, a única região com mediana positiva. A dependência sozinha não explica a dívida.

### 5.2 Múltiplas agregações: dívida por quintil de dependência

In [15]:
por_quintil = base.groupby('quintil_dep', observed=True).agg(
    municipios=('cod_ibge', 'count'),
    ufs=('uf', 'nunique'),
    populacao_mediana=('populacao', 'median'),
    dcl_rcl_media=('pct_dcl_rcl', 'mean'),
    dcl_rcl_mediana=('pct_dcl_rcl', 'median'),
    dcl_rcl_desvio=('pct_dcl_rcl', 'std'),
    dcl_rcl_max=('pct_dcl_rcl', 'max'),
    dc_rcl_mediana=('dc_rcl', 'median'),
)
por_quintil

,municipios,ufs,populacao_mediana,dcl_rcl_media,dcl_rcl_mediana,dcl_rcl_desvio,dcl_rcl_max,dc_rcl_mediana
quintil_dep,,,,,,,,
Q1 (menor),1097,26,"46,198.00",-1.65,-3.20,30.29,182.10,11.39
Q2,1097,25,"17,342.00",0.26,-2.41,28.72,244.21,8.95
Q3,1097,25,"12,329.00",5.28,-0.50,50.57,"1,333.33",10.05
Q4,1097,21,"7,486.00",2.88,-3.50,37.62,495.98,8.98
Q5 (maior),1097,16,"3,698.00",-7.99,-11.41,29.02,207.51,5.52


- A dívida é menor no grupo mais dependente: DCL/RCL mediana de **−11,4% no Q5**, contra −3,2% no Q1, e dívida bruta de 5,5% contra 11,4%. Mas a queda **não é contínua**: Q2 a Q4 são parecidos.
- **Média ≠ mediana** e desvio alto: poucos valores extremos puxam a média (o máximo do Q3, 1.333%, é provável erro de preenchimento).
- A população mediana cai de 46 mil (Q1) para 3,7 mil habitantes (Q5): os quintis também separam municípios por **porte**.

### 5.3 Múltiplos grupos: região × porte

In [16]:
regiao_porte = base.groupby(['regiao', 'porte'], observed=True).agg(
    dep_fpm=('dep_fpm', 'median'), dc_rcl=('dc_rcl', 'median'))

print('Dependência do FPM – mediana (%)')
display((regiao_porte['dep_fpm'] * 100).unstack())
print('Dívida bruta / RCL – mediana (%)')
display(regiao_porte['dc_rcl'].unstack())

Dependência do FPM – mediana (%)


porte,até 10 mil,10–20 mil,20–50 mil,50–100 mil,100–500 mil,> 500 mil
regiao,,,,,,
Norte,31.37,23.32,19.72,17.43,14.97,11.75
Nordeste,37.43,31.96,27.50,21.78,18.09,15.10
Centro-Oeste,29.85,21.61,17.91,15.56,10.67,5.74
Sudeste,41.39,31.41,23.36,16.84,11.12,2.87
Sul,35.04,25.75,19.91,14.73,11.55,4.25


Dívida bruta / RCL – mediana (%)


porte,até 10 mil,10–20 mil,20–50 mil,50–100 mil,100–500 mil,> 500 mil
regiao,,,,,,
Norte,8.06,8.17,6.07,11.19,7.99,39.85
Nordeste,12.02,19.39,18.59,17.18,25.18,32.98
Centro-Oeste,5.48,6.43,9.71,10.35,13.82,20.70
Sudeste,3.10,4.92,6.60,9.61,16.09,25.98
Sul,5.70,10.60,10.29,16.58,17.94,18.46


- **Em todas as regiões**, a dependência cai quando o porte aumenta, e a dívida bruta tende a subir.
- O **porte** é uma variável de confusão: parte do efeito "dependente → menos dívida" pode ser "pequeno → menos dívida".
- O Nordeste tem a maior dívida bruta em quase todas as faixas, até nos municípios pequenos.

### 5.4 Evolução no tempo: FPM por ano e região

In [17]:
fpm_ano = (fpm_longo.groupby(['ano', 'regiao'], observed=True)['fpm'].sum() / 1e9).unstack()
fpm_ano['Brasil'] = fpm_ano.sum(axis=1)
fpm_ano['variação (%)'] = fpm_ano['Brasil'].pct_change() * 100
fpm_ano.round(1)        # R$ bilhões, valores nominais

regiao,Norte,Nordeste,Centro-Oeste,Sudeste,Sul,Brasil,variação (%)
ano,,,,,,,
2015,6.00,24.30,4.80,21.30,11.90,68.30,NaN
2016,7.00,28.40,5.60,24.90,13.90,79.80,16.80
2017,6.80,27.60,5.40,24.20,13.50,77.50,-2.90
2018,7.40,29.30,5.80,25.90,14.50,82.90,6.90
2019,8.00,31.90,6.30,28.30,15.70,90.30,8.90
2020,7.70,30.50,6.10,27.20,15.10,86.50,-4.20
2021,10.30,40.90,8.10,36.30,20.20,115.70,33.80
2022,12.90,51.60,10.30,45.80,25.50,146.10,26.20
2023,13.40,53.70,10.70,47.60,26.50,151.80,3.90


O FPM nominal **quase triplicou** entre 2015 (R$ 68 bi) e 2025 (R$ 198 bi), com quedas só em 2017 e 2020 (pandemia) e forte alta em 2021–2022. Esse crescimento ajuda a explicar a folga de caixa (DCL negativa) da maioria dos municípios em 2025.

## 6. Indicadores

Cada indicador é uma **função** que recebe qualquer grupo de municípios: o Brasil, uma região, um quintil ou a seleção que o usuário fizer no sistema final.

| Indicador | Finalidade | Total considerado no % |
|---|---|---|
| **I1 – Dependência do FPM** | Medir quanto da receita vem do FPM | Municípios com FPM informado na DCA |
| **I2 – Situação da dívida** | Comparar a dívida com os limites legais | Municípios com DCL/RCL informada no RGF |
| **I3 – Acesso a crédito** | Testar o mecanismo da hipótese (dependentes tomam menos crédito) | Todos os municípios do grupo; a razão usa a soma das receitas do grupo |
| **I4 – Correlação dependência × dívida** | Resumir a hipótese em um número (negativo = confirma) | Municípios com os dois valores informados |

In [18]:
def ind_dependencia(df):
    dep = df['dep_fpm'].dropna()
    return pd.Series({'Dependência do FPM – mediana (%)': dep.median() * 100,
                      'Municípios com FPM ≥ 40% (%)': (dep >= LIMIAR_DEP).mean() * 100})

def ind_divida(df):
    dcl = df['pct_dcl_rcl'].dropna()
    return pd.Series({'DCL/RCL – mediana (%)': dcl.median(),
                      'DC/RCL – mediana (%)': df['dc_rcl'].median(),
                      'Municípios com DCL positiva (%)': (dcl > 0).mean() * 100,
                      f'Municípios acima de {LIMITE_ALERTA}% (%)': (dcl >= LIMITE_ALERTA).mean() * 100,
                      f'Municípios acima de {LIMITE_LEGAL}% (%)': (dcl > LIMITE_LEGAL).mean() * 100})

def ind_credito(df):
    return pd.Series({'Municípios com operação de crédito (%)': (df['op_credito'] > 0).mean() * 100,
                      'Operações de crédito / receita (%)': df['op_credito'].sum() / df['receita_total'].sum() * 100})

def ind_correlacao(df, y='dc_rcl'):
    # Spearman = correlação de Pearson entre as posições (rank); não é distorcida por extremos.
    par = df[['dep_fpm', y]].dropna()
    return par['dep_fpm'].rank().corr(par[y].rank())

def indicadores(df):
    return pd.concat([ind_dependencia(df), ind_divida(df), ind_credito(df)])

### I1, I2 e I3 por quintil de dependência

In [19]:
tabela_ind = base.groupby('quintil_dep', observed=True).apply(indicadores, include_groups=False)
tabela_ind.loc['Brasil'] = indicadores(base)
tabela_ind.T.round(1)

quintil_dep,Q1 (menor),Q2,Q3,Q4,Q5 (maior),Brasil
Dependência do FPM – mediana (%),15.90,23.80,29.40,35.80,45.40,29.40
Municípios com FPM ≥ 40% (%),0.00,0.00,0.00,0.00,97.90,19.60
DCL/RCL – mediana (%),-3.20,-2.40,-0.50,-3.50,-11.40,-4.20
DC/RCL – mediana (%),11.40,9.00,10.10,9.00,5.50,8.80
Municípios com DCL positiva (%),44.10,45.90,49.10,44.30,29.10,42.50
Municípios acima de 108% (%),0.30,0.50,1.60,1.20,0.40,0.80
Municípios acima de 120% (%),0.30,0.40,1.00,0.90,0.30,0.60
Municípios com operação de crédito (%),35.70,22.90,17.90,13.10,11.20,20.20
Operações de crédito / receita (%),1.40,0.40,0.40,0.20,0.20,1.10


- **I1:** no Brasil, o FPM é 29% da receita do município típico, e 1 em cada 5 municípios depende dele em 40% ou mais.
- **I2:** **57%** dos municípios não têm dívida líquida (caixa maior que a dívida). Só **0,8%** estão acima do alerta de 108%. No Q5, só 29% têm DCL positiva, contra 44% a 49% nos demais quintis.
- **I3:** é o indicador com o gradiente mais claro. A parcela de municípios que contrataram crédito cai de **36% (Q1) para 11% (Q5)**, o que confirma o mecanismo da hipótese.

### I4 – Correlação de Spearman por região

In [20]:
correlacao = pd.DataFrame({
    'dependência × DCL/RCL': base.groupby('regiao', observed=True).apply(ind_correlacao, y='pct_dcl_rcl', include_groups=False),
    'dependência × DC/RCL':  base.groupby('regiao', observed=True).apply(ind_correlacao, y='dc_rcl', include_groups=False),
})
correlacao.loc['Brasil'] = [ind_correlacao(base, 'pct_dcl_rcl'), ind_correlacao(base, 'dc_rcl')]
correlacao.round(2)

,dependência × DCL/RCL,dependência × DC/RCL
regiao,,
Norte,-0.10,-0.09
Nordeste,0.03,0.10
Centro-Oeste,0.06,-0.01
Sudeste,-0.25,-0.30
Sul,-0.28,-0.36
Brasil,-0.10,-0.12


**I4:** no Brasil, a correlação é **negativa, porém fraca** (−0,10 a −0,12). Ela é mais forte no Sul e no Sudeste (−0,25 a −0,36) e chega a ser **positiva no Nordeste**. A hipótese vale em parte do país, mas não em todo ele.

## 7. Comparação de desempenho: filtrar antes × depois do `merge`

**Operação:** obter o FPM mensal de 2025 dos municípios com DCL/RCL acima de 30%. A condição usa as duas tabelas: `fpm_longo` (735 mil linhas) e `base` (5,5 mil linhas).

In [21]:
indicadores_mun = base[['cod_siafi', 'pct_dcl_rcl', 'dep_fpm']]

def merge_depois_filtra():
    m = fpm_longo.merge(indicadores_mun, on='cod_siafi')
    return m[(m['ano'] == 2025) & (m['pct_dcl_rcl'] > 30)]

def filtra_depois_merge():
    return fpm_longo[fpm_longo['ano'] == 2025].merge(indicadores_mun[indicadores_mun['pct_dcl_rcl'] > 30], on='cod_siafi')

# 1) Equivalência: mesmas linhas, colunas e valores
ordenar = lambda df: df.sort_values(['cod_siafi', 'mes']).reset_index(drop=True)
pd.testing.assert_frame_equal(ordenar(merge_depois_filtra()), ordenar(filtra_depois_merge()))
print(f'Resultados equivalentes: {fmt(len(filtra_depois_merge()))} linhas')

# 2) Tempos: 1 execução de aquecimento + 10 medições de cada, com os mesmos dados
def medir(funcao, repeticoes=10):
    funcao()
    tempos = []
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        funcao()
        tempos.append((time.perf_counter() - inicio) * 1000)
    return pd.Series(tempos)

tempos = pd.DataFrame({'merge → filtro': medir(merge_depois_filtra), 'filtro → merge': medir(filtra_depois_merge)})
resultado = tempos.agg(['min', 'median', 'mean', 'std']).T.add_suffix(' (ms)')
resultado['vezes mais lento'] = resultado['median (ms)'] / resultado['median (ms)'].min()
resultado

Resultados equivalentes: 7.656 linhas


,min (ms),median (ms),mean (ms),std (ms),vezes mais lento
merge → filtro,213.76,227.67,281.54,92.76,13.38
filtro → merge,16.35,17.01,16.93,0.36,1.00


- **`filtro → merge` foi de 8 a 13 vezes mais rápido** em todas as execuções de teste (a tabela acima mostra a execução atual). Os tempos exatos mudam a cada execução e conforme a máquina, mas a proporção se mantém, e o desvio baixo mostra que as medições foram estáveis.
- **Por quê:** o `merge → filtro` cruza as 735 mil linhas e depois descarta 99% delas. O `filtro → merge` reduz antes os dois lados (66 mil linhas de 2025 e cerca de 640 municípios), e o `merge`, que é a operação mais cara (casa chaves e copia colunas), processa só uma fração dos dados.
- **Regra para o sistema:** filtrar sempre antes de cruzar.

## 8. Relatório consolidado

Reúne os indicadores **por região** (ordenados pela dependência do FPM), com uma linha **Brasil**, e **por quintil**. O Excel tem quatro abas, incluindo a **lista de alerta** e a **base municipal** que o sistema final vai usar.

In [22]:
def resumo(df):
    return pd.concat([pd.Series({'Municípios': len(df), 'População (milhões)': df['populacao'].sum() / 1e6}),
                      indicadores(df),
                      pd.Series({'Spearman dependência × DC/RCL': ind_correlacao(df),
                                 'FPM 2025 (R$ bilhões)': df['fpm_2025'].sum() / 1e9})])

relatorio = (base.groupby('regiao', observed=True).apply(resumo, include_groups=False)
                 .sort_values('Dependência do FPM – mediana (%)', ascending=False))
relatorio.loc['Brasil'] = resumo(base)
relatorio_quintil = base.groupby('quintil_dep', observed=True).apply(resumo, include_groups=False)

relatorio.T.round(2)

regiao,Sudeste,Nordeste,Sul,Centro-Oeste,Norte,Brasil
Municípios,"1,660.00","1,771.00","1,175.00",463.00,421.00,"5,490.00"
População (milhões),86.59,55.03,30.53,13.54,17.09,202.78
Dependência do FPM – mediana (%),31.48,31.01,28.95,23.24,23.07,29.40
Municípios com FPM ≥ 40% (%),28.59,17.18,17.12,11.45,10.00,19.58
DCL/RCL – mediana (%),-11.34,12.50,-14.16,-8.10,-1.33,-4.19
DC/RCL – mediana (%),5.12,16.78,8.13,6.99,7.99,8.78
Municípios com DCL positiva (%),28.49,74.18,18.31,30.74,46.06,42.52
Municípios acima de 108% (%),0.06,2.05,0.00,0.22,0.95,0.77
Municípios acima de 120% (%),0.06,1.42,0.00,0.22,0.95,0.57
Municípios com operação de crédito (%),22.35,8.92,40.09,17.28,6.41,20.16


In [23]:
alerta = (base.loc[base['pct_dcl_rcl'] >= LIMITE_ALERTA, ['cod_ibge', 'municipio', 'uf', 'populacao', 'pct_dcl_rcl', 'dep_fpm']]
              .sort_values('pct_dcl_rcl', ascending=False))

arquivo = SAIDA / 'relatorio_endividamento_municipal_2025.xlsx'
with pd.ExcelWriter(arquivo) as excel:
    relatorio.round(2).to_excel(excel, sheet_name='por_regiao')
    relatorio_quintil.round(2).to_excel(excel, sheet_name='por_quintil')
    alerta.to_excel(excel, sheet_name='lista_alerta', index=False)
    base.to_excel(excel, sheet_name='base_municipal', index=False)
relatorio.round(2).to_csv(SAIDA / 'relatorio_por_regiao_2025.csv', sep=';', decimal=',', encoding='utf-8-sig')

# Confere a exportação relendo o arquivo
print({aba: df.shape for aba, df in pd.read_excel(arquivo, sheet_name=None).items()})

{'por_regiao': (6, 14), 'por_quintil': (5, 14), 'lista_alerta': (42, 6), 'base_municipal': (5490, 31)}


## 9. Análise e interpretação dos resultados

### 1. Principais informações identificadas
1. **A hipótese se confirma só em parte.** O quintil mais dependente (Q5) tem a menor dívida (DCL/RCL mediana de −11,4% e dívida bruta de 5,5% da RCL, contra −3,2% e 11,4% no Q1). Mas a correlação é fraca (−0,10 a −0,12), e Q2 a Q4 são parecidos.
2. **O acesso a crédito explica o mecanismo.** A parcela de municípios que contrataram crédito cai de forma contínua, de 36% (Q1) para 11% (Q5).
3. **A região muda o resultado.** Nordeste e Sudeste dependem igualmente do FPM, mas o Nordeste é o mais endividado (74% com DCL positiva) e concentra 150 dos 167 municípios pequenos endividados. Lá, a correlação é positiva.
4. **O endividamento excessivo é localizado.** 57% dos municípios têm caixa maior que a dívida; só 42 (0,8%) passam do alerta de 108% e 31 (0,6%) do limite de 120%.

### 2. Contribuição dos agrupamentos
- **Por quintil:** mostrou que a diferença se concentra no grupo mais dependente e que a média é distorcida por extremos, algo que a correlação sozinha esconderia.
- **Por região:** revelou a principal exceção à hipótese (Nordeste × Sudeste), que a média nacional escondia.
- **Região × porte:** mostrou que o porte do município influencia ao mesmo tempo a dependência e a dívida.
- **Por ano:** mostrou o crescimento do FPM, que explica a folga de caixa de 2025.

### 3. Indicadores mais importantes
- **I1 (dependência)** e **I2 (situação da dívida)** são essenciais: I1 é a variável da hipótese, e I2 liga a análise aos limites legais e gera a lista de alerta.
- **I3 (acesso a crédito)** teve o resultado mais claro e explica o porquê da relação.
- **I4 (correlação)** deve ser mostrado **por recorte** (região, porte), pois o valor nacional esconde comportamentos opostos.

### 4. Limitações
- **Abrangência:** 5.490 de 5.569 municípios (sem o DF).
  - 34 só entregaram o RGF, 15 só a DCA e 29 nenhum dos dois;
  - 17 não informaram a DCL.
- **Dado autodeclarado:** valores como 1.333% (Pontal/SP) indicam erro de preenchimento. A mediana e o Spearman reduzem o efeito.
- **Ausente = zero:** validado para a dívida pela identidade contábil. Para as operações de crédito, segue a lógica do demonstrativo.
- **Retrato de um ano:** o dado da dívida é só de 2025, então mostra associação, não causalidade. O porte ainda não foi controlado.
- **Valores nominais:** o FPM não foi corrigido pela inflação. O FPM do Tesouro é líquido e o da DCA é bruto.
- **De-para por nome:** depende de 22 correções manuais, validadas pela razão Tesouro/DCA.

### 5. Uso no projeto final
- **Dados:** a aba `base_municipal` será a tabela principal do sistema.
- **Filtros como opções de consulta:** região/UF (Filtro 1), faixa de dependência (Filtro 2) e lista de alerta (Filtro 3).
- **Funções reaproveitáveis:** `ind_dependencia`, `ind_divida`, `ind_credito`, `ind_correlacao` e `resumo` calculam os indicadores para qualquer seleção do usuário.
- **Desempenho:** as consultas seguirão a regra de filtrar antes de cruzar.
- **Próximas etapas:**
  1. incluir o PIB per capita (IBGE);
  2. comparar municípios de mesmo porte;
  3. corrigir o FPM pela inflação;
  4. criar as visualizações.